# OpenDC Demo 5: Temporal Shifting 

This is the third of three demos focusing on sustainability techniques. In the previous two demos, we looked at the impact of horizontal scaling and batteries. In this demo, we will conclude by looking at temporal shifting.

Green energy is often based on environmental phenomena. This means that its availability can vary significantly over time. For example, solar energy might be abundant during a sunny day, but is not available at night.

Datacenters could use this fact by scheduling tasks to align with the availability of low-carbon energy, which is called temporal shifting. However, temporal shifting is not without challenges. Delaying tasks too much will reduce the quality of service (QoS) for users.

In this demo, we will explore the trade-off between carbon emissions and QoS introduced by temporal shifting.

The experiments shown in this demo are simplified versions of the experiments run for the paper [OpenDC-STEAM](https://ieeexplore.ieee.org/document/11619111). 


## Workload Traces

To use temporal shifting in OpenDC, you first need to define which tasks can be delayed, and by how much. 
In OpenDC, we do this using the `deferrable` and `deadline` columns of the `tasks.parquet` file in the workload traces. 

Let's look at the workload traces again.

In [6]:
import pandas as pd

df_tasks = pd.read_parquet("workload_traces/surf_month/tasks.parquet")
df_fragments = pd.read_parquet("workload_traces/surf_month/fragments.parquet")

df_tasks.head()

,id,submission_time,duration,cpu_count,cpu_capacity,mem_capacity,deferrable,deadline
0,1519830,1656604920000,1000,16,33600.0,97,True,1656691320000
1,1519833,1656604949000,1000,16,33600.0,97,True,1656691349000
2,1520988,1656605328000,315000,16,33600.0,97,True,1656691728000
3,1520994,1656605793000,330000,16,33600.0,97,True,1656692193000
4,1521049,1656608110000,248000,16,33600.0,97,True,1656694510000


As you can see, each task defines whether it is `deferrable` and what its `deadline` is.
The deadline indicates the latest time the task can finish. 

In this example, all tasks can be delayed, and the deadline is 1 day after the optimal deadline. 

# Experiment

Temporal shifting can be enabled by defining a scheduler in the `allocationPolicies` input. [experiments/5.temporal_shifting/shifting_experiment.json](experiments/5.temporal_shifting/shifting_experiment.json) defines an experiment in which two different schedulers are used: one normal scheduler and one carbon-aware scheduler. 

This is done by the following snippet:
```json
"allocationPolicies": [
    {   
        "type": "filter",
        "filters": [
            {
                "type": "compute"
            },
            {
                "type": "vcpu",
                "allocationRatio": 1.0
            },
            {
                "type": "ram",
                "allocationRatio": 1.5
            }
        ],
        "weighers": [
            {
                "type": "ram",
                "multiplier": 1.0
            }
        ]
    },
    {   
        "type": "filter",
        "filters": [
            {
                "type": "compute"
            },
            {
                "type": "vcpu",
                "allocationRatio": 1.0
            },
            {
                "type": "ram",
                "allocationRatio": 1.5
            }
        ],
        "weighers": [
            {
                "type": "ram",
                "multiplier": 1.0
            }
        ],
        "timeshift": {
            "shortForecastThreshold": 0.2,
            "longForecastThreshold": 0.35,
            "taskStopper": { "forecastThreshold": 0.4 }
        }
    }
],
```

OpenDC provides two types of allocation policies (or schedulers): prefab or filter. 
Prefabs are predefined filter schedulers created for easy use. 

A normal filter scheduler selects a host using the following steps: 
1. It *filters* out all the hosts that are not able to run the task.
2. It *sorts* the hosts based on a number of weights. 

Besides the normal scheduling policy, OpenDC supports temporal shifting (which is the main focus of this demo).

When temporal shifting is used, the scheduler will delay tasks if:
1. The current carbon intensity is higher than a threshold. This threshold is based on a carbon forecast. The scheduler uses different thresholds for short (<2 hours) and long tasks, given by `shortForecastThreshold` and `longForecastThreshold`, respectively. 
2. The task can be delayed further. 

Additionally, we can use a `taskStopper`. This will interrupt a task when the carbon intensity gets higher than the `forecastThreshold`. 

For more information about allocation policies (with and without temporal shifting), see [Allocation Policy](https://opendc.org/learn/documentation/Input/AllocationPolicy).

# Running OpenDC

In [8]:
import subprocess

pathToExperiment = "experiments/5.temporal_shifting/shifting_experiment.json"
result = subprocess.run(["OpenDCExperimentRunner/bin/opendc", "run", pathToExperiment])

Starting replay


WARN  
					Metrics after 24 hours:
						Tasks Total: 1172
						Tasks Active: 139
						Tasks Pending: 64
						Tasks Completed: 969
						Tasks Terminated: 0

WARN  
					Metrics after 24 hours:
						Tasks Total: 1172
						Tasks Active: 139
						Tasks Pending: 64
						Tasks Completed: 969
						Tasks Terminated: 0

WARN  
					Metrics after 24 hours:
						Tasks Total: 1172
						Tasks Active: 139
						Tasks Pending: 64
						Tasks Completed: 969
						Tasks Terminated: 0

WARN  
					Metrics after 48 hours:
						Tasks Total: 1413
						Tasks Active: 30
						Tasks Pending: 0
						Tasks Completed: 1383
						Tasks Terminated: 0

WARN  
					Metrics after 48 hours:
						Tasks Total: 1413
						Tasks Active: 30
						Tasks Pending: 0
						Tasks Completed: 1383
						Tasks Terminated: 0

WARN  
					Metrics after 48 hours:
						Tasks Total: 1413
						Tasks Active: 30
						Tasks Pending: 0
						Tasks Completed: 1383
						Tasks Terminated: 0

WARN  
					Metrics after 72 ho

Starting replay


WARN  
					Metrics after 24 hours:
						Tasks Total: 1172
						Tasks Active: 136
						Tasks Pending: 0
						Tasks Completed: 1036
						Tasks Terminated: 0

WARN  
					Metrics after 24 hours:
						Tasks Total: 1172
						Tasks Active: 136
						Tasks Pending: 0
						Tasks Completed: 1036
						Tasks Terminated: 0

WARN  
					Metrics after 24 hours:
						Tasks Total: 1172
						Tasks Active: 136
						Tasks Pending: 0
						Tasks Completed: 1036
						Tasks Terminated: 0

WARN  
					Metrics after 48 hours:
						Tasks Total: 1413
						Tasks Active: 29
						Tasks Pending: 0
						Tasks Completed: 1384
						Tasks Terminated: 0

WARN  
					Metrics after 48 hours:
						Tasks Total: 1413
						Tasks Active: 29
						Tasks Pending: 0
						Tasks Completed: 1384
						Tasks Terminated: 0

WARN  
					Metrics after 48 hours:
						Tasks Total: 1413
						Tasks Active: 29
						Tasks Pending: 0
						Tasks Completed: 1384
						Tasks Terminated: 0

WARN  
					Metrics after 72 ho

┌─────────┬──────┬──────┬──────────┬──────────┬────────┬───────────┬──────────┐
│ Scenario│ Seed │ Tasks│ Completed│ Terminate│ Mean CP│ Energy [kW│ Carbon [k│
├─────────┼──────┼──────┼──────────┼──────────┼────────┼───────────┼──────────┤
│ #0      │ 0    │ 47117│ 47117    │ 0        │ 18.8%  │ 4612293.35│ 1573769.6│
├─────────┼──────┼──────┼──────────┼──────────┼────────┼───────────┼──────────┤
│ #1      │ 0    │ 47117│ 47117    │ 0        │ 18.9%  │ 4677724.50│ 1541216.8│
└─────────┴──────┴──────┴──────────┴──────────┴────────┴───────────┴──────────┘
Simulated 2 run(s); Parquet results in output


WARN  
					Metrics after 744 hours:
						Tasks Total: 47117
						Tasks Active: 73
						Tasks Pending: 0
						Tasks Completed: 47044
						Tasks Terminated: 0

WARN  
					Metrics after 744 hours:
						Tasks Total: 47117
						Tasks Active: 73
						Tasks Pending: 0
						Tasks Completed: 47044
						Tasks Terminated: 0

WARN  
					Metrics after 744 hours:
						Tasks Total: 47117
						Tasks Active: 73
						Tasks Pending: 0
						Tasks Completed: 47044
						Tasks Terminated: 0



## Output

After the simulation has run, the output folder should contain two subfolders, one for each scenario. 
If you are unsure about which output is connected to which scenario, you can use the `trackr.json` file in the meta folder.


**Exercise 1:** 
Load the results into Pandas DataFrames and give them appropriate names.

In [9]:
# Your code goes here

<details>
<summary>Click to reveal the answer to Exercise 1</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


df_host_base = pd.read_parquet("output/5.temporal_shifting/raw-output/0/seed=0/host.parquet")
df_powerSource_base = pd.read_parquet("output/5.temporal_shifting/raw-output/0/seed=0/powerSource.parquet")
df_task_base = pd.read_parquet("output/5.temporal_shifting/raw-output/0/seed=0/task.parquet")
df_service_base = pd.read_parquet("output/5.temporal_shifting/raw-output/0/seed=0/service.parquet")

df_host_shifted = pd.read_parquet("output/5.temporal_shifting/raw-output/1/seed=0/host.parquet")
df_powerSource_shifted = pd.read_parquet("output/5.temporal_shifting/raw-output/1/seed=0/powerSource.parquet")
df_task_shifted = pd.read_parquet("output/5.temporal_shifting/raw-output/1/seed=0/task.parquet")
df_service_shifted = pd.read_parquet("output/5.temporal_shifting/raw-output/1/seed=0/service.parquet")
```

</details>

## Aggregating results

Let's start again by looking at some aggregated results. We would like to understand how temporal shifting impacts the performance and sustainability of the datacenter. 
To do that, print metrics related to performance and sustainability.

We suggest printing the following metrics:
1. runtime
2. energy usage
3. carbon emission

However, there might be more interesting metrics to explore. See the [Output](https://opendc.org/learn/documentation/Output) page on our website to understand which metrics could be interesting.

**Exercise 2:** 
Explore the impact of temporal shifting by printing some aggregated metrics.

In [ ]:
# Put your code here

<details>
<summary>Click to reveal the answer to Exercise 2</summary>

```python
# runtime
runtime_base = df_service_base.timestamp.max() / 3_600_000 # convert to hours
runtime_shifted = df_service_shifted.timestamp.max() / 3_600_000

print(f"Normal scheduling took {runtime_base:.2f} hours to complete the workload")
print(f"Carbon-aware scheduling took {runtime_shifted:.2f} hours to complete the workload\n")


# energy
energy = df_powerSource_base.energy_usage.max() / 3_600_000 # convert energy to kWh
energy_shifted = df_powerSource_shifted.energy_usage.max() / 3_600_000 # convert energy to kWh

print(f"Normal scheduling used {energy:.2f} kWh during the workload")
print(f"Carbon-aware scheduling used {energy_shifted:.2f} kWh during the workload\n")

# carbon
carbon = df_powerSource_base.carbon_emission.max() / 1000
carbon_shifted = df_powerSource_shifted.carbon_emission.max() / 1000

print(f"Normal scheduling emitted {carbon:.2f} kg CO2 during the workload")
print(f"Carbon-aware scheduling emitted {carbon_shifted:.2f} kg CO2 during the workload")
print(f"This is a decrease of {((carbon - carbon_shifted) / carbon) * 100:.2f}%\n")
```

</details>

## Service Level Agreements

The aggregated results show that although temporal shifting can reduce the carbon emissions of a datacenter, it can also result in lower performance. 
OpenDC exports the delay of a task using the `scheduling_delay` metric, which defines the amount of time a task was pending before being scheduled. 

**Exercise 3:** 
Calculate the average delay of the tasks for each scheduler.

*Remember:* OpenDC exports multiple samples for each task. You will have to aggregate the samples from a task to get its final scheduling delay.

In [17]:
# Your code goes here

<details>
<summary>Click to reveal the answer to Exercise 3</summary>

```python
delays_base = df_task_base.groupby('task_id')['scheduling_delay'].agg('max') / 60_000 # Convert to minutes
delays_shifted = df_task_shifted.groupby('task_id')['scheduling_delay'].agg('max') / 60_000

print(f"The average scheduling delay with a normal scheduler is:      {delays_base.mean():.2f} minutes")
print(f"The average scheduling delay when using temporal shifting is: {delays_shifted.mean():.2f} minutes")
```

</details>

# Visualization

To better understand how temporal shifting helps reduce carbon emissions, we can compare its behavior over time. 

We suggest plotting the following metrics:
1. tasks active
2. energy usage
3. carbon emission

It may also help to plot the carbon intensity in some way.

**Exercise 4:** 
Use visualization to better understand the impact of temporal shifting.

Can you use your visualization to explain how temporal shifting can reduce carbon emissions?

In [26]:
# Your code goes here

<details>
<summary>Click to reveal the answer to Exercise 4</summary>

```python
# active tasks
fig, (ax0, ax1) = plt.subplots(2,1, height_ratios = [2,1], figsize=(10,5), sharex=True, gridspec_kw={'hspace': 0.1})
ax0.plot(df_service_base.timestamp / 3_600_000, df_service_base.tasks_active, label="Normal")
ax0.plot(df_service_shifted.timestamp / 3_600_000, df_service_shifted.tasks_active, label="Carbon-Aware")

ax0.legend(loc="upper left", fontsize=10)

ax0.set_title("Tasks active during a month-long surfsara workload", fontsize=15)
ax0.set_ylabel("Tasks active", fontsize=13)
ax0.xaxis.set_major_locator(plt.MaxNLocator(10))

ax1.plot(df_service_shifted.timestamp / 3_600_000, df_powerSource_shifted.carbon_intensity, label="Shifted")

ax1.set_xlabel("Time [hours]", fontsize=13)
ax1.set_ylabel("Carbon Intensity", fontsize=13)

plt.show()

# Energy
fig, (ax0, ax1) = plt.subplots(2,1, height_ratios = [2,1], figsize=(10,5), sharex=True, gridspec_kw={'hspace': 0.1})
ax0.plot(df_powerSource_base.timestamp / 3_600_000, df_powerSource_base.power_draw / 1000, label="Normal")
ax0.plot(df_powerSource_shifted.timestamp / 3_600_000, df_powerSource_shifted.power_draw / 1000, label="Carbon-Aware")

ax0.legend(loc="upper left", fontsize=10)

ax0.set_title("Power Draw during a month-long surfsara workload", fontsize=15)
ax0.set_ylabel("Power Draw (kW)", fontsize=13)
ax0.xaxis.set_major_locator(plt.MaxNLocator(10))

ax1.plot(df_powerSource_shifted.timestamp / 3_600_000, df_powerSource_shifted.carbon_intensity, label="Shifted")
ax1.set_xlabel("Time [hours]", fontsize=13)
ax1.set_ylabel("Carbon Intensity", fontsize=13)

plt.show()

# Carbon
df_powerSource_base["carbon_diff"] = df_powerSource_base.carbon_emission - df_powerSource_base.carbon_emission.shift(1, fill_value=0)
df_powerSource_shifted["carbon_diff"] = df_powerSource_shifted.carbon_emission - df_powerSource_shifted.carbon_emission.shift(1, fill_value=0)

fig, (ax0, ax1) = plt.subplots(2,1, height_ratios = [2,1], figsize=(10,5), sharex=True, gridspec_kw={'hspace': 0.1})
ax0.plot(df_powerSource_base.timestamp / 3_600_000, df_powerSource_base.carbon_diff / 1000, label="Normal")
ax0.plot(df_powerSource_shifted.timestamp / 3_600_000, df_powerSource_shifted.carbon_diff / 1000, label="Carbon-Aware")

ax0.legend(loc="upper left", fontsize=10)

ax0.set_title("Carbon emissions during a month-long surfsara workload", fontsize=15)
ax0.set_ylabel("Carbon Emissions (kg)", fontsize=13)
ax0.xaxis.set_major_locator(plt.MaxNLocator(10))

ax1.plot(df_powerSource_shifted.timestamp / 3_600_000, df_powerSource_shifted.carbon_intensity, label="Shifted")

ax1.set_xlabel("Time [hours]", fontsize=13)
ax1.set_ylabel("Carbon Intensity", fontsize=13)

plt.show()
```

</details>

## Next steps

In this demo, we used OpenDC to understand the impact of temporal shifting on the carbon footprint of a datacenter. We showed that temporal shifting can help reduce carbon emissions, but can also impact the QoS of users.

What else could you do with OpenDC regarding temporal shifting?
We suggest looking at:
1. What is the best shifting setup for the Netherlands?
2. What happens when we switch to another location?

For inspiration, have a look at the temporal shifting section in the [OpenDC-STEAM](https://ieeexplore.ieee.org/document/11619111) paper. 